# GiBUU Data Preparation

This notebook prepares data for training the interaction and propagation models.

**Workflow:**
1. **Configuration** (always run first)
2. **ROOT → H5** (optional - skip if H5 files exist)
3. **Process H5 → Pairs** (can start here if H5 exists)
   - 3.1: Extract sequences
   - 3.2: Extract pairs
   - 3.3: Filter by changes
   - 3.4: Save to NPZ
- **Test** (verify NPZ loading; calculate statistics; plot distributions)
- **Training Integration** (examples for next steps)

**Output:**
- `npz_with_changes/*.npz`: Interaction pairs (for transformer)
- `npz_without_changes/*.npz`: Propagation pairs (for propagation model)


## Step 1: Configuration

**Always run this cell first** - defines job IDs and paths for all subsequent steps.


In [ ]:
# Import libraries
import numpy as np
import pickle
import glob
from pathlib import Path
from typing import List

# Import from package
from gibuu_transformer.data_processing import (
    build_gibuu_h5_from_root,
    extract_particle_sequences,
    extract_timestep_pairs,
    filter_pairs_with_changes,
    prepare_interaction_data,
    prepare_propagation_data
)

print("✓ Libraries imported successfully!")

# ============================================================================
# CONFIGURATION - Modify this for your experiment
# ============================================================================

# Job IDs to process
JOB_IDS = [990]  # Example: [990] or list(range(990, 1000))

# Data paths
ROOT_BASE_PATH = "/exp/dune/data/users/yinrui/GiBUU/GPTdata"
H5_BASE_PATH = "/exp/dune/data/users/yinrui/GiBUU/GPTdata"
OUTPUT_PATH = "./processed_data"

# Processing options
TIMESTEPS = range(1, 203)  # Time steps 1-202 (skip 1 in extract_particle_sequences)
GROUP_KEY = "perturbative"
MAX_PARTICLES = 200

# Convert to list if range
if not isinstance(JOB_IDS, list):
    JOB_IDS = list(JOB_IDS)

# Create output directories
Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)
(Path(OUTPUT_PATH) / "npz_with_changes").mkdir(exist_ok=True)
(Path(OUTPUT_PATH) / "npz_without_changes").mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  Job IDs: {JOB_IDS}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Max particles: {MAX_PARTICLES}")


## Step 2: Convert ROOT to H5 (Optional)

**Can skip this step** if H5 files already exist - jump to Step 3.


In [ ]:
# Convert ROOT → H5
h5_files = []

for job_id in JOB_IDS:
    data_path = f"{ROOT_BASE_PATH}/job.{job_id:06d}"
    h5_path = f"{H5_BASE_PATH}/GiBUU_FSI_{job_id:06d}.h5"
    
    if Path(h5_path).exists():
        print(f"Job {job_id:06d}: H5 exists, skipping")
        h5_files.append(h5_path)
        continue
    
    if not Path(data_path).exists():
        print(f"Job {job_id:06d}: WARNING - ROOT not found: {data_path}")
        continue
    
    print(f"Job {job_id:06d}: Converting...")
    try:
        build_gibuu_h5_from_root(
            data_path=data_path,
            out_path=h5_path,
            timesteps=TIMESTEPS,
            group_key=GROUP_KEY
        )
        h5_files.append(h5_path)
        print(f"Job {job_id:06d}: ✓ Complete")
    except Exception as e:
        print(f"Job {job_id:06d}: ERROR - {e}")

print(f"\n✓ H5 files ready: {len(h5_files)}")

# Alternative: If H5 exists, uncomment to load directly:
# h5_files = [f"{H5_BASE_PATH}/GiBUU_FSI_{job_id:06d}.h5" for job_id in JOB_IDS]
# h5_files = [f for f in h5_files if Path(f).exists()]
# print(f"Loaded {len(h5_files)} existing H5 files")


## Step 3: Process H5 → Pairs

**Can start here** if H5 files already exist (Step 2 skipped).

This step has 3 sub-steps that depend on each other (run sequentially):
- 3.1: Extract sequences from H5
- 3.2: Extract consecutive time step pairs
- 3.3: Filter pairs by particle changes

### Step 3.1: Extract Sequences

In [ ]:
# If h5_files not defined (skipped Step 2), load from JOB_IDS
if 'h5_files' not in globals():
    h5_files = [f"{H5_BASE_PATH}/GiBUU_FSI_{job_id:06d}.h5" for job_id in JOB_IDS]
    h5_files = [f for f in h5_files if Path(f).exists()]
    print(f"Loaded {len(h5_files)} existing H5 files")

all_sequences = []

for h5_file in h5_files:
    print(f"Extracting: {Path(h5_file).name}")
    sequences = extract_particle_sequences(h5_file, gr_key=GROUP_KEY)
    all_sequences.extend(sequences)
    print(f"  → {len(sequences)} events")

print(f"\n✓ Total sequences: {len(all_sequences)}")

# Statistics
non_empty = [s for s in all_sequences if len(s) > 0]
if len(non_empty) > 0:
    lengths = [len(s) for s in non_empty]
    print(f"  Non-empty: {len(non_empty)}")
    print(f"  Length - min: {min(lengths)}, max: {max(lengths)}, avg: {np.mean(lengths):.1f}")


### Step 3.2: Extract Pairs


In [ ]:
pairs = extract_timestep_pairs(all_sequences, max_particles=MAX_PARTICLES)

print(f"✓ Extracted {len(pairs)} time step pairs")

if len(pairs) > 0:
    input_sizes = [len(inp) for inp, _ in pairs]
    output_sizes = [len(out) for _, out in pairs]
    print(f"  Input - min: {min(input_sizes)}, max: {max(input_sizes)}, avg: {np.mean(input_sizes):.1f}")
    print(f"  Output - min: {min(output_sizes)}, max: {max(output_sizes)}, avg: {np.mean(output_sizes):.1f}")


### Step 3.3: Filter Pairs


In [ ]:
pairs_with_changes, pairs_without_changes = filter_pairs_with_changes(pairs)

print(f"✓ Filtering complete:")
print(f"  Total: {len(pairs)}")
print(f"  With changes (interactions): {len(pairs_with_changes)} ({100*len(pairs_with_changes)/len(pairs):.1f}%)")
print(f"  Without changes (propagation): {len(pairs_without_changes)} ({100*len(pairs_without_changes)/len(pairs):.1f}%)")

# Example
if len(pairs_with_changes) > 0:
    inp, out = pairs_with_changes[0]
    print(f"\nExample interaction: {len(inp)} → {len(out)} particles")

if len(pairs_without_changes) > 0:
    inp, out = pairs_without_changes[0]
    print(f"Example propagation: {len(inp)} → {len(out)} particles (same types)")


### Step 3.4: Save to NPZ


In [ ]:
# Helper function to convert pairs to NPZ
def pairs_to_npz(pairs_list):
    if len(pairs_list) == 0:
        return {}
    
    max_input = max(len(inp) for inp, _ in pairs_list)
    max_output = max(len(out) for _, out in pairs_list)
    n_pairs = len(pairs_list)
    
    input_data = np.zeros((n_pairs, max_input, 11), dtype=np.float32)
    output_data = np.zeros((n_pairs, max_output, 11), dtype=np.float32)
    input_lengths = np.zeros(n_pairs, dtype=np.int32)
    output_lengths = np.zeros(n_pairs, dtype=np.int32)
    
    for i, (inp, out) in enumerate(pairs_list):
        input_lengths[i] = len(inp)
        output_lengths[i] = len(out)
        for j, particle in enumerate(inp):
            input_data[i, j] = particle
        for j, particle in enumerate(out):
            output_data[i, j] = particle
    
    return {
        'input_data': input_data,
        'output_data': output_data,
        'input_lengths': input_lengths,
        'output_lengths': output_lengths
    }

# Save interaction pairs
with_dir = Path(OUTPUT_PATH) / "npz_with_changes"
for job_id in JOB_IDS:
    npz_path = with_dir / f"pairs_with_changes_{job_id:06d}.npz"
    if len(pairs_with_changes) > 0:
        arrays = pairs_to_npz(pairs_with_changes)
        np.savez_compressed(npz_path, **arrays)
        size_mb = npz_path.stat().st_size / 1e6
        print(f"✓ Saved: {npz_path.name} ({size_mb:.1f} MB, {len(pairs_with_changes)} pairs)")
    break  # One file per run

# Save propagation pairs
without_dir = Path(OUTPUT_PATH) / "npz_without_changes"
for job_id in JOB_IDS:
    npz_path = without_dir / f"pairs_without_changes_{job_id:06d}.npz"
    if len(pairs_without_changes) > 0:
        arrays = pairs_to_npz(pairs_without_changes)
        np.savez_compressed(npz_path, **arrays)
        size_mb = npz_path.stat().st_size / 1e6
        print(f"✓ Saved: {npz_path.name} ({size_mb:.1f} MB, {len(pairs_without_changes)} pairs)")
    break

print(f"\n✓ NPZ files saved to: {OUTPUT_PATH}")
print("  Format: (n_pairs, max_particles, 11) + lengths arrays")
print("  Token: [ts, gibuuID, charge, x, y, z, m, E, Px, Py, Pz]")


---

## Test

Verify NPZ loading, calculate feature statistics, and visualize distributions.


In [ ]:
# Find and load NPZ files
with_files = sorted(glob.glob(str(Path(OUTPUT_PATH) / "npz_with_changes" / "*.npz")))
without_files = sorted(glob.glob(str(Path(OUTPUT_PATH) / "npz_without_changes" / "*.npz")))

print(f"Found {len(with_files)} interaction files")
print(f"Found {len(without_files)} propagation files")

# Test loading
if len(with_files) > 0:
    print(f"\nLoading: {Path(with_files[0]).name}")
    data = np.load(with_files[0])
    print(f"✓ Loaded successfully")
    print(f"  Arrays: {list(data.keys())}")
    print(f"  input_data: {data['input_data'].shape}")
    print(f"  output_data: {data['output_data'].shape}")
    print(f"  Pairs: {len(data['input_lengths'])}")
    
    # Example
    idx = 0
    inp_len = data['input_lengths'][idx]
    out_len = data['output_lengths'][idx]
    print(f"\nExample pair:")
    print(f"  Input: {inp_len} particles")
    print(f"  Output: {out_len} particles")
    print(f"  First particle: {data['input_data'][idx, 0, :inp_len]}")
    print(f"  Format: [ts, gibuuID, charge, x, y, z, m, E, Px, Py, Pz]")

print("\n✓ Test passed!")

# ============================================================================
# Calculate Feature Statistics
# ============================================================================

# Calculate FEATS_MEAN, FEATS_SIGMA from all particles
print("\n" + "="*80)
print("CALCULATING FEATURE STATISTICS")
print("="*80)
print("Calculating feature statistics from processed data...")

all_features = []
all_deltas = []

# Collect from pairs_without_changes (propagation pairs)
for inp, out in pairs_without_changes:
    for particle in inp:
        # Token: [ts, gibuuID, charge, x, y, z, m, E, Px, Py, Pz]
        ts, gibuu_id, charge, x, y, z, m, E, Px, Py, Pz = particle
        features = [x, y, z, E-m, Px, Py, Pz]  # [x, y, z, KE, Px, Py, Pz]
        all_features.append(features)
    
    # Calculate deltas for propagation pairs
    if len(inp) == len(out):
        for p_in, p_out in zip(inp, out):
            feat_in = np.array([p_in[3], p_in[4], p_in[5], p_in[7]-p_in[6], p_in[8], p_in[9], p_in[10]])
            feat_out = np.array([p_out[3], p_out[4], p_out[5], p_out[7]-p_out[6], p_out[8], p_out[9], p_out[10]])
            delta = feat_out - feat_in
            all_deltas.append(delta)

# Also collect from pairs_with_changes (interaction pairs, for complete statistics)
for inp, out in pairs_with_changes:
    for particle in inp + out:
        ts, gibuu_id, charge, x, y, z, m, E, Px, Py, Pz = particle
        features = [x, y, z, E-m, Px, Py, Pz]
        all_features.append(features)

# Convert to arrays
all_features = np.array(all_features)
all_deltas = np.array(all_deltas) if len(all_deltas) > 0 else np.array([])

# Calculate statistics
FEATS_MEAN_COMPUTED = np.mean(all_features, axis=0).tolist()
FEATS_SIGMA_COMPUTED = np.std(all_features, axis=0).tolist()

print(f"\n{'='*80}")
print("FEATURE STATISTICS (Absolute Values)")
print('='*80)
print(f"FEATS_MEAN = {FEATS_MEAN_COMPUTED}")
print(f"FEATS_SIGMA = {FEATS_SIGMA_COMPUTED}")
print(f"\nFeatures: [x, y, z, KE, Px, Py, Pz] where KE = E - m")
print(f"Total particles: {len(all_features)}")

# Calculate delta statistics
if len(all_deltas) > 0:
    FEATS_DELTA_MEAN_COMPUTED = np.mean(all_deltas, axis=0).tolist()
    FEATS_DELTA_SIGMA_COMPUTED = np.std(all_deltas, axis=0).tolist()
    
    # Also calculate filtered statistics (ignore near-zero for E/p)
    FEATS_DELTA_MEAN_FILTERED = []
    FEATS_DELTA_SIGMA_FILTERED = []
    
    tolerance = 1e-10
    feature_names = ['Δx', 'Δy', 'Δz', 'ΔKE', 'ΔPx', 'ΔPy', 'ΔPz']
    
    print(f"\n{'='*80}")
    print("DELTA STATISTICS (Δ Values)")
    print('='*80)
    
    for i in range(7):
        feat_deltas = all_deltas[:, i]
        
        # For position (0,1,2): use all values
        # For energy/momentum (3,4,5,6): filter near-zero
        if i < 3:
            mask = np.ones(len(feat_deltas), dtype=bool)
        else:
            mask = np.abs(feat_deltas) > tolerance
        
        filtered = feat_deltas[mask]
        filtered_mean = np.mean(filtered) if len(filtered) > 0 else 0.0
        filtered_sigma = np.std(filtered) if len(filtered) > 0 else 1.0
        
        FEATS_DELTA_MEAN_FILTERED.append(filtered_mean)
        FEATS_DELTA_SIGMA_FILTERED.append(filtered_sigma)
        
        pct_kept = 100 * len(filtered) / len(feat_deltas)
        print(f"{feature_names[i]:>4}: mean={FEATS_DELTA_MEAN_COMPUTED[i]:10.6f}, sigma={FEATS_DELTA_SIGMA_COMPUTED[i]:10.6f} (all)")
        print(f"      mean={filtered_mean:10.6f}, sigma={filtered_sigma:10.6f} (filtered, {pct_kept:.1f}% kept)")
    
    print(f"\n{'='*80}")
    print("RECOMMENDED VALUES (use in constants.py or notebooks):")
    print('='*80)
    print(f"FEATS_DELTA_MEAN = {FEATS_DELTA_MEAN_FILTERED}")
    print(f"FEATS_DELTA_SIGMA = {FEATS_DELTA_SIGMA_FILTERED}")
    print(f"\nTotal transitions: {len(all_deltas)}")
else:
    print("\nNo delta statistics (no propagation pairs found)")


In [ ]:
import matplotlib.pyplot as plt

feature_names = ['x [fm]', 'y [fm]', 'z [fm]', 'KE [GeV]', 'Px [GeV]', 'Py [GeV]', 'Pz [GeV]']
delta_names = ['Δx [fm]', 'Δy [fm]', 'Δz [fm]', 'ΔKE [GeV]', 'ΔPx [GeV]', 'ΔPy [GeV]', 'ΔPz [GeV]']

# Plot absolute features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(7):
    ax = axes[i]
    feat_values = all_features[:, i]
    
    # Use percentiles to handle outliers
    p1, p99 = np.percentile(feat_values, [1, 99])
    filtered = feat_values[(feat_values >= p1) & (feat_values <= p99)]
    
    ax.hist(filtered, bins=100, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(FEATS_MEAN_COMPUTED[i], color='red', linestyle='--', linewidth=2, label=f'Mean={FEATS_MEAN_COMPUTED[i]:.3f}')
    ax.axvline(FEATS_MEAN_COMPUTED[i] + FEATS_SIGMA_COMPUTED[i], color='orange', linestyle=':', linewidth=1.5, label=f'±σ')
    ax.axvline(FEATS_MEAN_COMPUTED[i] - FEATS_SIGMA_COMPUTED[i], color='orange', linestyle=':', linewidth=1.5)
    ax.set_xlabel(feature_names[i], fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{feature_names[i]} Distribution\nμ={FEATS_MEAN_COMPUTED[i]:.3f}, σ={FEATS_SIGMA_COMPUTED[i]:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[7].axis('off')
plt.suptitle('Absolute Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot delta features
if len(all_deltas) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(7):
        ax = axes[i]
        delta_values = all_deltas[:, i]
        
        # Use percentiles to handle outliers
        p1, p99 = np.percentile(delta_values, [1, 99])
        filtered = delta_values[(delta_values >= p1) & (delta_values <= p99)]
        
        ax.hist(filtered, bins=100, alpha=0.7, edgecolor='black', linewidth=0.5, color='green')
        ax.axvline(FEATS_DELTA_MEAN_COMPUTED[i], color='red', linestyle='--', linewidth=2, label=f'Mean={FEATS_DELTA_MEAN_COMPUTED[i]:.4f}')
        ax.axvline(FEATS_DELTA_MEAN_FILTERED[i], color='blue', linestyle='--', linewidth=2, label=f'Filtered μ={FEATS_DELTA_MEAN_FILTERED[i]:.4f}')
        ax.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
        ax.set_xlabel(delta_names[i], fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'{delta_names[i]} Distribution\nσ(all)={FEATS_DELTA_SIGMA_COMPUTED[i]:.4f}, σ(filt)={FEATS_DELTA_SIGMA_FILTERED[i]:.4f}', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    axes[7].axis('off')
    plt.suptitle('Delta Feature Distributions (Propagation Pairs)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Distribution plots complete!")
else:
    print("\nNo delta distributions (no propagation pairs)")


---

## Training Integration

Examples for loading processed data in training notebooks (`prepare_interaction_data` and `prepare_propagation_data`).


In [ ]:
# ============================================================================
# EXAMPLE 1: Prepare for INTERACTION model (GiBUU_interaction.ipynb)
# ============================================================================

# # Load NPZ and convert to pairs
# npz_files = glob.glob('./processed_data/npz_with_changes/*.npz')
# 
# pairs_loaded = []
# for npz_file in npz_files:
#     data = np.load(npz_file)
#     for i in range(len(data['input_lengths'])):
#         inp_len = data['input_lengths'][i]
#         out_len = data['output_lengths'][i]
#         inp = [list(token) for token in data['input_data'][i, :inp_len]]
#         out = [list(token) for token in data['output_data'][i, :out_len]]
#         pairs_loaded.append((inp, out))
# 
# # Prepare for transformer training
# from gibuu_transformer.data_processing import prepare_interaction_data
# 
# interaction_data = prepare_interaction_data(
#     pairs=pairs_loaded,
#     save_stats_path='feature_stats.json'
# )
# 
# print(f"✓ Prepared {len(interaction_data['input_encoded_ids'])} interaction sequences")


# ============================================================================
# EXAMPLE 2: Prepare for PROPAGATION model (GiBUU_propagation.ipynb)
# ============================================================================

# # Helper to load pairs from NPZ
# def load_pairs_from_npz(npz_files):
#     pairs = []
#     for npz_file in npz_files:
#         data = np.load(npz_file)
#         for i in range(len(data['input_lengths'])):
#             inp_len = data['input_lengths'][i]
#             out_len = data['output_lengths'][i]
#             inp = [list(token) for token in data['input_data'][i, :inp_len]]
#             out = [list(token) for token in data['output_data'][i, :out_len]]
#             pairs.append((inp, out))
#     return pairs
# 
# # Load both types
# with_changes_files = glob.glob('./processed_data/npz_with_changes/*.npz')
# without_changes_files = glob.glob('./processed_data/npz_without_changes/*.npz')
# 
# pairs_with_changes_loaded = load_pairs_from_npz(with_changes_files)
# pairs_without_changes_loaded = load_pairs_from_npz(without_changes_files)
# 
# # Prepare for propagation training
# from gibuu_transformer.data_processing import prepare_propagation_data
# 
# propagation_data = prepare_propagation_data(
#     pairs_without_changes=pairs_without_changes_loaded,
#     pairs_with_changes=pairs_with_changes_loaded,
#     stats_path='feature_stats.json'
# )
# 
# print(f"✓ Prepared {len(propagation_data['particle_types'])} propagation steps")

print("Uncomment examples above to use in training")
